# 01 — Data audit and preparation

**Purpose.** Inspect the raw bankruptcy dataset, validate the core fields, encode the target, and create the modelling dataset used by later notebooks.

**Inputs.** `data/raw/american_bankruptcy.csv`.

**Outputs.** Processed model data, raw-data summaries, target distribution, feature dictionary, and validation report.

**What you will learn.** How the raw labels become the binary failure target and how the financial predictors are kept separate from identifiers and the target.

**Run first.** This is the first notebook in the workflow.

## Imports and paths

Notebook 01 starts from the raw CSV, checks the data structure, encodes the bankruptcy target, and saves the model-ready table.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """Find the project root from the current working directory."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        has_structure = (
            (candidate / "notebooks").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "README.md").is_file()
        )
        if has_structure:
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run the notebook from the repository root or notebooks directory."
    )


PROJECT_ROOT = find_project_root()
project_root = PROJECT_ROOT

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "american_bankruptcy.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
REPORT_DIR = PROJECT_ROOT / "outputs" / "report"
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / "model_dataset.csv"

for directory in [PROCESSED_DATA_DIR, TABLES_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


## Dataset constants

These names define the raw identifiers, target labels, and readable financial-variable labels used in the audit.

In [2]:
COMPANY_COLUMN = "company_name"
RAW_TARGET_COLUMN = "status_label"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

ALIVE_LABEL = "alive"
FAILED_LABEL = "failed"

FEATURE_NAME_MAP = {
    "X1": "Current assets",
    "X2": "Cost of goods sold",
    "X3": "Depreciation and amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net income",
    "X7": "Total receivables",
    "X8": "Market value",
    "X9": "Net sales",
    "X10": "Total assets",
    "X11": "Total long-term debt",
    "X12": "EBIT",
    "X13": "Gross profit",
    "X14": "Total current liabilities",
    "X15": "Retained earnings",
    "X16": "Total revenue",
    "X17": "Total liabilities",
    "X18": "Total operating expenses",
}

FEATURE_DESCRIPTION_MAP = {
    "X1": "Assets expected to be sold, converted into cash, or used within one year.",
    "X2": "Direct costs related to producing or selling the firm's goods and services.",
    "X3": "Depreciation of tangible assets and amortization of intangible assets.",
    "X4": "Earnings before interest, taxes, depreciation, and amortization.",
    "X5": "Goods and raw materials held by the firm for production or sale.",
    "X6": "Profit after expenses and costs have been deducted from revenue.",
    "X7": "Money owed to the firm by customers for delivered goods or services.",
    "X8": "Market capitalization or market value of the publicly traded company.",
    "X9": "Gross sales minus returns, allowances, and discounts.",
    "X10": "Total assets owned or controlled by the company.",
    "X11": "Debt obligations due after more than one year.",
    "X12": "Earnings before interest and taxes.",
    "X13": "Profit after subtracting costs related to manufacturing and selling.",
    "X14": "Short-term obligations due within one year.",
    "X15": "Accumulated profits retained in the business after dividends and losses.",
    "X16": "Total income from sales before subtracting expenses.",
    "X17": "Total debts and obligations owed to outside parties.",
    "X18": "Expenses incurred through normal business operations.",
}


## Validation helpers

The audit fails early if a required column, target label, or feature definition is missing.

In [3]:
def validate_required_columns(data: pd.DataFrame) -> None:
    """Check that the raw identifier, year, and status columns are present."""
    missing = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")


def validate_target_labels(data: pd.DataFrame) -> None:
    """Check that the raw status column uses only the expected labels."""
    observed = set(data[RAW_TARGET_COLUMN].dropna().unique())
    unexpected = observed.difference({ALIVE_LABEL, FAILED_LABEL})
    if unexpected:
        raise ValueError(f"Unexpected target labels found: {sorted(unexpected)}")


def identify_feature_columns(data: pd.DataFrame) -> list[str]:
    """Return numeric financial variables, excluding identifiers and labels."""
    excluded = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}
    return [
        column
        for column in data.columns
        if column not in excluded and pd.api.types.is_numeric_dtype(data[column])
    ]


## Load raw data

Start with the original CSV and show the basic shape before any modelling decisions are made.

In [4]:
raw_data = pd.read_csv(RAW_DATA_PATH)

raw_overview = pd.DataFrame(
    {
        "item": ["rows", "columns", "companies", "first_year", "last_year"],
        "value": [
            len(raw_data),
            raw_data.shape[1],
            raw_data[COMPANY_COLUMN].nunique(),
            raw_data[YEAR_COLUMN].min(),
            raw_data[YEAR_COLUMN].max(),
        ],
    }
)
display(raw_overview)
display(raw_data.head())

,item,value
0,rows,78682
1,columns,21
2,companies,8971
3,first_year,1999
4,last_year,2018


,company_name,status_label,year,X1,X2,X3,X4,X5,X6,X7,...,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18
0,C_1,alive,1999,511.267,833.107,18.373,89.031,336.018,35.163,128.348,...,1024.333,740.998,180.447,70.658,191.226,163.816,201.026,1024.333,401.483,935.302
1,C_1,alive,2000,485.856,713.811,18.577,64.367,320.590,18.531,115.187,...,874.255,701.854,179.987,45.790,160.444,125.392,204.065,874.255,361.642,809.888
2,C_1,alive,2001,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,...,638.721,710.199,217.699,4.711,112.244,150.464,139.603,638.721,399.964,611.514
3,C_1,alive,2002,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,...,606.337,686.621,164.658,3.573,109.590,203.575,124.106,606.337,391.633,575.592
4,C_1,alive,2003,432.204,523.302,26.680,47.491,247.245,3.504,104.661,...,651.958,709.292,248.666,20.811,128.656,131.261,131.884,651.958,407.608,604.467


## Check required columns and labels

These assertions replace the most important raw-data validation tests from the package version.

In [5]:
validate_required_columns(raw_data)
validate_target_labels(raw_data)

required_columns_check = pd.DataFrame(
    {"required_column": [COMPANY_COLUMN, YEAR_COLUMN, RAW_TARGET_COLUMN], "present": True}
)
target_label_check = raw_data[RAW_TARGET_COLUMN].value_counts().rename_axis("status_label").reset_index(name="count")
target_label_check["share"] = target_label_check["count"] / target_label_check["count"].sum()

display(required_columns_check)
display(target_label_check)

,required_column,present
0,company_name,True
1,year,True
2,status_label,True


,status_label,count,share
0,alive,73462,0.933657
1,failed,5220,0.066343


## Identify financial predictors

The predictor set is the original `X1` through `X18` financial variables. Company, year, and target fields stay out of the model matrix.

In [6]:
feature_columns = identify_feature_columns(raw_data)
expected_feature_columns = [f"X{i}" for i in range(1, 19)]
assert feature_columns == expected_feature_columns, f"Unexpected predictors: {feature_columns}"

feature_preview = pd.DataFrame(
    {
        "feature": feature_columns,
        "readable_name": [FEATURE_NAME_MAP[feature] for feature in feature_columns],
    }
)
display(feature_preview)

,feature,readable_name
0,X1,Current assets
1,X2,Cost of goods sold
2,X3,Depreciation and amortization
3,X4,EBITDA
4,X5,Inventory
5,X6,Net income
6,X7,Total receivables
7,X8,Market value
8,X9,Net sales
9,X10,Total assets


## Summarize raw-data quality

These files document the raw input before target encoding and model preparation.

In [7]:
summary = pd.DataFrame(
    [
        {
            "n_rows": int(len(raw_data)),
            "n_columns": int(raw_data.shape[1]),
            "n_companies": int(raw_data[COMPANY_COLUMN].nunique()),
            "min_year": int(raw_data[YEAR_COLUMN].min()),
            "max_year": int(raw_data[YEAR_COLUMN].max()),
            "n_feature_columns": int(len(feature_columns)),
            "n_failed_rows": int(raw_data[RAW_TARGET_COLUMN].eq(FAILED_LABEL).sum()),
            "n_alive_rows": int(raw_data[RAW_TARGET_COLUMN].eq(ALIVE_LABEL).sum()),
            "failure_rate": float(raw_data[RAW_TARGET_COLUMN].eq(FAILED_LABEL).mean()),
            "n_missing_values": int(raw_data.isna().sum().sum()),
            "n_duplicate_rows": int(raw_data.duplicated().sum()),
        }
    ]
)

target_distribution = raw_data[RAW_TARGET_COLUMN].value_counts().rename_axis("status_label").reset_index(name="count")
target_distribution["share"] = target_distribution["count"] / target_distribution["count"].sum()

validation_report = {
    "n_rows": int(len(raw_data)),
    "n_columns": int(raw_data.shape[1]),
    "columns": list(raw_data.columns),
    "feature_columns": feature_columns,
    "n_feature_columns": len(feature_columns),
    "target_counts": {str(k): int(v) for k, v in raw_data[RAW_TARGET_COLUMN].value_counts().items()},
    "company_count": int(raw_data[COMPANY_COLUMN].nunique()),
    "year_min": int(raw_data[YEAR_COLUMN].min()),
    "year_max": int(raw_data[YEAR_COLUMN].max()),
    "missing_values_by_column": {str(k): int(v) for k, v in raw_data.isna().sum().items()},
    "n_duplicate_rows": int(raw_data.duplicated().sum()),
}

summary.to_csv(TABLES_DIR / "data_summary.csv", index=False)
target_distribution.to_csv(TABLES_DIR / "target_distribution.csv", index=False)
with (REPORT_DIR / "raw_data_validation.json").open("w", encoding="utf-8") as file:
    json.dump(validation_report, file, indent=2)

display(summary)
display(target_distribution)

,n_rows,n_columns,n_companies,min_year,max_year,n_feature_columns,n_failed_rows,n_alive_rows,failure_rate,n_missing_values,n_duplicate_rows
0,78682,21,8971,1999,2018,18,5220,73462,0.066343,0,0


,status_label,count,share
0,alive,73462,0.933657
1,failed,5220,0.066343


## Encode the target

The raw text label is converted to a binary target: alive observations are `0`, failed observations are `1`.

In [8]:
data = raw_data.copy()
data[TARGET_COLUMN] = data[RAW_TARGET_COLUMN].map({ALIVE_LABEL: 0, FAILED_LABEL: 1}).astype(int)

assert set(data[TARGET_COLUMN].unique()) == {0, 1}, "Encoded target must be binary with values 0 and 1."
encoded_target_check = data[[RAW_TARGET_COLUMN, TARGET_COLUMN]].drop_duplicates().sort_values(TARGET_COLUMN)
display(encoded_target_check)

,status_label,failed
0,alive,0
50,failed,1


## Remove constant predictors

A constant feature carries no information for classification. The original data have no retained modelling value in such columns, so they are removed if present.

In [9]:
constant_columns = [column for column in feature_columns if data[column].nunique(dropna=False) <= 1]
retained_features = [column for column in feature_columns if column not in constant_columns]

constant_feature_check = pd.DataFrame(
    {
        "n_candidate_features": [len(feature_columns)],
        "n_removed_constant_features": [len(constant_columns)],
        "removed_features": [", ".join(constant_columns) if constant_columns else "None"],
        "n_retained_features": [len(retained_features)],
    }
)
display(constant_feature_check)

,n_candidate_features,n_removed_constant_features,removed_features,n_retained_features
0,18,0,None,18


## Create and save the model dataset

The modelling table keeps company and year for leakage-safe splitting and diagnostics, but only `X1`-`X18` enter the predictor matrix later.

In [10]:
model_dataset = data[[COMPANY_COLUMN, YEAR_COLUMN, TARGET_COLUMN, *retained_features]].copy()

assert COMPANY_COLUMN in model_dataset.columns and YEAR_COLUMN in model_dataset.columns, "Identifiers must be preserved."
assert RAW_TARGET_COLUMN not in model_dataset.columns, "Raw target label must not remain in modelling data."
assert TARGET_COLUMN not in retained_features, "Target must not be part of predictors."
assert len(model_dataset) == len(raw_data), "Preprocessing must preserve row count."

model_dataset.to_csv(MODEL_DATASET_PATH, index=False)
display(model_dataset.head())
display(pd.DataFrame({"output": [str(MODEL_DATASET_PATH.relative_to(project_root))], "rows": [len(model_dataset)], "columns": [model_dataset.shape[1]]}))

,company_name,year,failed,X1,X2,X3,X4,X5,X6,X7,...,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18
0,C_1,1999,0,511.267,833.107,18.373,89.031,336.018,35.163,128.348,...,1024.333,740.998,180.447,70.658,191.226,163.816,201.026,1024.333,401.483,935.302
1,C_1,2000,0,485.856,713.811,18.577,64.367,320.590,18.531,115.187,...,874.255,701.854,179.987,45.790,160.444,125.392,204.065,874.255,361.642,809.888
2,C_1,2001,0,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,...,638.721,710.199,217.699,4.711,112.244,150.464,139.603,638.721,399.964,611.514
3,C_1,2002,0,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,...,606.337,686.621,164.658,3.573,109.590,203.575,124.106,606.337,391.633,575.592
4,C_1,2003,0,432.204,523.302,26.680,47.491,247.245,3.504,104.661,...,651.958,709.292,248.666,20.811,128.656,131.261,131.884,651.958,407.608,604.467


,output,rows,columns
0,data/processed/model_dataset.csv,78682,21


## Create and save the feature dictionary

Readable names and short descriptions make later tables and figures easier to interpret without changing the original feature codes.

In [11]:
feature_dictionary = pd.DataFrame(
    {
        "feature": retained_features,
        "readable_name": [FEATURE_NAME_MAP.get(feature, feature) for feature in retained_features],
        "description": [FEATURE_DESCRIPTION_MAP.get(feature, "") for feature in retained_features],
        "role": "financial_predictor",
        "included_in_model": True,
    }
)
if constant_columns:
    removed = pd.DataFrame(
        {
            "feature": constant_columns,
            "readable_name": [FEATURE_NAME_MAP.get(feature, feature) for feature in constant_columns],
            "description": [FEATURE_DESCRIPTION_MAP.get(feature, "") for feature in constant_columns],
            "role": "removed_constant_column",
            "included_in_model": False,
        }
    )
    feature_dictionary = pd.concat([feature_dictionary, removed], ignore_index=True)

feature_dictionary.to_csv(TABLES_DIR / "feature_dictionary.csv", index=False)
assert pd.read_csv(MODEL_DATASET_PATH).shape == model_dataset.shape, "Saved model dataset shape must match in-memory data."
display(feature_dictionary)

,feature,readable_name,description,role,included_in_model
0,X1,Current assets,"Assets expected to be sold, converted into cas...",financial_predictor,True
1,X2,Cost of goods sold,Direct costs related to producing or selling t...,financial_predictor,True
2,X3,Depreciation and amortization,Depreciation of tangible assets and amortizati...,financial_predictor,True
3,X4,EBITDA,"Earnings before interest, taxes, depreciation,...",financial_predictor,True
4,X5,Inventory,Goods and raw materials held by the firm for p...,financial_predictor,True
5,X6,Net income,Profit after expenses and costs have been dedu...,financial_predictor,True
6,X7,Total receivables,Money owed to the firm by customers for delive...,financial_predictor,True
7,X8,Market value,Market capitalization or market value of the p...,financial_predictor,True
8,X9,Net sales,"Gross sales minus returns, allowances, and dis...",financial_predictor,True
9,X10,Total assets,Total assets owned or controlled by the company.,financial_predictor,True


## Summary

- Checked the raw bankruptcy file for the expected company, year, status, and financial-variable columns.
- Encoded the target as `0 = alive` and `1 = failed`, then removed any constant predictors before modelling.
- Saved the prepared modelling file to `data/processed/model_dataset.csv` and saved audit tables in `outputs/tables/`.
- Notebook 02 starts from this prepared modelling file and creates the company-level train/test split.
